# VoiceLK — speaker_01 VITS training on Kaggle

Trains a single-speaker VITS model on the speaker_01 recordings with
`model_training/train_speaker.py`, from the Kaggle dataset `pamindudasunpriya/tts-s01`:

```
tts_s01/
├── speaker_01_metadata.txt   wavs/sin_s1_0001.wav|<IPA sequence>   (one line per clip)
└── audio/
    ├── sin_s1_0001.wav
    └── ...
```

The metadata already holds the IPA (made with `data_preparation/prepare_speaker_dataset.py`),
so the notebook keeps it as it is and only converts the audio to one sample rate, mono.

**Before running**

1. *Add Input* → the `tts-s01` dataset. For a resumed run, also add the previous run's
   notebook output (see section 5).
2. *Settings* → Accelerator **GPU T4 x2**, Internet **on**.
3. If the GitHub repository is private, add a secret named `GITHUB_TOKEN`
   (*Add-ons* → *Secrets*) holding a read-only personal access token.
4. Run unattended with *Save Version* → **Save & Run All (Commit)**. An interactive session
   stops when the browser goes idle; a committed run keeps going for up to 12 hours and
   keeps everything written under `/kaggle/working`.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/RootCode-AI/voicelk_ml.git"
BRANCH = "Tharinda/newDev_V1.0"

SPEAKER = "speaker_01"

# The uploaded dataset: IPA metadata plus the folder holding the wav files
DATASET_DIR = "/kaggle/input/datasets/pamindudasunpriya/tts-s01/tts_s01"
METADATA_IN = f"{DATASET_DIR}/{SPEAKER}_metadata.txt"
AUDIO_DIR = f"{DATASET_DIR}/audio"
SAMPLE_RATE = 22050

# Fine-tune: a .pth checkpoint under /kaggle/input to start from (e.g. the custom run's
# best_model.pth). Recommended while the recording set is small - VITS from scratch needs
# hours of audio. When starting from the custom run, set SAMPLE_RATE = 16000 to match it.
# Leave empty to train from scratch.
RESTORE_FROM = ""

# Resume: a folder under /kaggle/input holding an earlier output of this notebook
# (the newest run folder inside it is picked). Leave empty for a fresh run.
RESUME_FROM = ""

EPOCHS = 1000
BATCH_SIZE = 16

WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/voicelk_ml"
DATA_DIR = f"{WORK}/data"
RUNS_DIR = f"{WORK}/runs/{SPEAKER}"

## 2. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 3. Code — clone the repository and switch to `BRANCH`

Clones on the first run; on a re-run it fetches and switches to `BRANCH`, so a changed
`BRANCH` or newly pushed commits are always picked up.

In [ ]:
import os, subprocess

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret; cloning anonymously (works for a public repository)")
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL


def git(*args):
    # Commands are not echoed, so the token in auth_url never reaches the saved output
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.replace(token, "***") if token else result.stderr)
    return result.stdout.strip()


# 1. Clone (only the first time)
if not os.path.exists(f"{REPO_DIR}/.git"):
    print(f"Cloning {REPO_URL} ...")
    git("clone", auth_url, REPO_DIR)

# 2. Fetch the branch, switch to it and bring it up to date with the remote
git("-C", REPO_DIR, "fetch", auth_url, f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}")
git("-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}")
git("-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}")

print("Branch:", git("-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"))
print("Commit:", git("-C", REPO_DIR, "log", "--oneline", "-1"))

### Dependencies

Installs the training requirements into this kernel's Python, skipping `torch` and
`torchaudio` so the Kaggle image's CUDA build is kept. Output is shown and a failed install
stops the notebook, so a missing package surfaces here and not halfway through training.
The `TTS` and `trainer` packages are vendored in the repository, not installed.

In [ ]:
import re, subprocess, sys

KEEP_FROM_IMAGE = {"torch", "torchaudio"}

with open(f"{REPO_DIR}/model_training/requirements.txt", encoding="utf-8") as f:
    requirements = [line.strip() for line in f if line.strip() and not line.lstrip().startswith("#")]
to_install = [r for r in requirements if re.split(r"[<>=!~;\[ ]", r)[0] not in KEEP_FROM_IMAGE]
print("Installing:", " ".join(to_install))

result = subprocess.run([sys.executable, "-m", "pip", "install", "--no-input", *to_install])
if result.returncode != 0:
    raise RuntimeError("pip install failed - see the output above")

In [ ]:
# Every module the training and data-preparation scripts import, by import name
import importlib

MODULES = [
    "torch", "torchaudio", "numpy", "scipy", "pandas", "sklearn", "soundfile", "librosa",
    "numba", "tqdm", "coqpit", "psutil", "fsspec", "tensorboard", "yaml", "aiohttp",
    "packaging", "mutagen", "pysbd", "matplotlib", "inflect", "anyascii", "Cython",
    "eng_to_ipa", "sinling",
]
missing = []
for name in MODULES:
    try:
        module = importlib.import_module(name)
        print(f"  ok  {name:12s} {getattr(module, '__version__', '')}")
    except Exception as e:
        missing.append(name)
        print(f"  !!  {name:12s} {type(e).__name__}: {e}")
assert not missing, f"Missing modules: {missing} - re-run the install cell and read its errors"

In [ ]:
# The vendored TTS/trainer packages and the text front-end, imported the way the training
# script imports them (from model_training/, with model_engine/ on the path)
checks = {
    "model_training": "from trainer import Trainer; from TTS.tts.models.vits import Vits; import vocab_utils; "
                      "print('TTS + trainer + vocab_utils: ok')",
    "model_engine": "from pipeline import TextProcessingPipeline as P; "
                    "print('front-end:', P().process('මේ Flutter course එකට')['ipa_sequence'])",
}
for folder, script in checks.items():
    subprocess.run([sys.executable, "-c", script], cwd=f"{REPO_DIR}/{folder}", check=True)

## 4. Data

In [ ]:
import glob

assert os.path.exists(METADATA_IN), f"{METADATA_IN} not found - is the tts-s01 dataset added as input?"
wavs = glob.glob(f"{AUDIO_DIR}/*.wav")
with open(METADATA_IN, encoding="utf-8") as f:
    lines = [line for line in f if line.strip()]
print(f"Metadata: {len(lines)} lines | Audio: {len(wavs)} wav files in {AUDIO_DIR}")
print(lines[0][:150])

In [ ]:
METADATA = f"{SPEAKER}_metadata.txt"
!cd {REPO_DIR} && python data_preparation/prepare_speaker_dataset.py \
    --metadata "{METADATA_IN}" --wav-dir "{AUDIO_DIR}" --wav-out "{DATA_DIR}/wavs" \
    --sample-rate {SAMPLE_RATE} --out "{DATA_DIR}/{METADATA}"

## 5. Fine-tune / resume (optional)

- **Fine-tune** (`RESTORE_FROM`): starts a new run from an existing checkpoint's weights.
  Layers whose shape differs (e.g. the text embedding, if the symbol set changed) start fresh.
- **Resume** (`RESUME_FROM`): Kaggle stops a session after 12 hours. Open this notebook's
  *Output* tab from the finished version, add it as input (*Add Input* → *Your Work*),
  set `RESUME_FROM` to its folder and commit again. The run folder is copied into
  `/kaggle/working` because inputs are read-only. Resume takes precedence over fine-tune.

In [ ]:
import shutil

extra_args = ""
if RESUME_FROM:
    runs = sorted(
        (os.path.dirname(p) for p in glob.glob(f"{RESUME_FROM}/**/config.json", recursive=True)),
        key=lambda d: max(os.path.getmtime(os.path.join(d, f)) for f in os.listdir(d)),
    )
    assert runs, f"No run folder with config.json under {RESUME_FROM}"
    target = f"{RUNS_DIR}/{os.path.basename(runs[-1])}"
    if not os.path.exists(target):
        shutil.copytree(runs[-1], target)
    extra_args = f'--continue_path "{target}"'
    print("Resuming", target)
elif RESTORE_FROM:
    assert os.path.exists(RESTORE_FROM), f"{RESTORE_FROM} not found"
    extra_args = f'--restore_path "{RESTORE_FROM}"'
    print("Fine-tuning from", RESTORE_FROM)
else:
    print("Fresh run from scratch")

## 6. Train

In [ ]:
!cd {REPO_DIR} && python model_training/train_speaker.py \
    --data-path "{DATA_DIR}" --metadata {METADATA} \
    --run-name voicelk_vits_{SPEAKER} --output-path "{RUNS_DIR}" \
    --epochs {EPOCHS} --batch-size {BATCH_SIZE} {extra_args}

## 7. Output

The run folder under `/kaggle/working/runs` holds `config.json`, `best_model.pth`, the last
two `checkpoint_*.pth` and the TensorBoard logs. The converted audio is removed so the saved
output stays small - it is rebuilt from the dataset on every run.

In [ ]:
shutil.rmtree(f"{DATA_DIR}/wavs", ignore_errors=True)
for run in glob.glob(f"{RUNS_DIR}/*/"):
    print(run)
    for f in sorted(os.listdir(run)):
        print(f"  {f:45s} {os.path.getsize(os.path.join(run, f)) / 1e6:9.1f} MB")